#### Concept 1: Single inheritance

a child class automatically inherits everything the parent already defined, and can add new attributes/methods on top of that, without touching the parent's code at all.

In [1]:
class Account:
    def __init__(self, owner, balance):
        self.owner = owner
        self.balance = balance

    def deposit(self, amount):
        self.balance += amount

class SavingsAccount(Account):
    def __init__(self, owner, balance, interest_rate):
        super().__init__(owner, balance)     # reuse Account's setup
        self.interest_rate = interest_rate    # add something new

    def add_interest(self):                   # only SavingsAccount has this
        self.balance += self.balance * self.interest_rate

s = SavingsAccount("Kavya", 1000, 0.05)

s.deposit(500)
s.add_interest()

print(s.owner, s.balance, s.interest_rate)

Kavya 1575.0 0.05


s.__dict__ has all three — owner, balance, interest_rate — merged into one single dict, exactly as you said. It doesn't matter that interest_rate was set up by SavingsAccount.__init__ while owner/balance came from Account.__init__ via super() — once both __init__ calls finish, it's all just sitting in s's one dict together, no separation by "which class set this up."


deposit genuinely lives only in Account.__dict__ (confirmed: True/False above) — the code is only written once, up in the parent, never copied into SavingsAccount.

But when you call s.deposit(500), self inside that code is s itself — type(self) prints SavingsAccount, not Account. So self.balance += amount reaches into s's one real dict, the same one holding interest_rate too.


#### Method overriding: child redefines a method the parent already has

In [4]:
class Account:
    def __init__(self, owner, balance):
        self.owner = owner
        self.balance = balance

    def deposit(self, amount):
        print("  Account.deposit running")
        self.balance += amount

class SavingsAccount(Account):
    def __init__(self, owner, balance, interest_rate):
        super().__init__(owner, balance)
        self.interest_rate = interest_rate

    def deposit(self, amount):   # OVERRIDING - same method name as the parent
        print("  SavingsAccount.deposit running (adds a $10 bonus)")
        self.balance += amount + 10

acc1 = Account("Rahul", 500)
s1 = SavingsAccount("Kavya", 1000, 0.05)

acc1.deposit(100)
s1.deposit(100)

print("acc1.balance:", acc1.balance)
print("s1.balance:", s1.balance)

  Account.deposit running
  SavingsAccount.deposit running (adds a $10 bonus)
acc1.balance: 600
s1.balance: 1110


#### Extending vs. replacing with super().method()

In [5]:
class Account:
    def __init__(self, owner, balance):
        self.owner = owner
        self.balance = balance

    def deposit(self, amount):
        # imagine this got updated later to add a transaction fee
        fee = 2
        self.balance += (amount - fee)
        print(f"  Account.deposit: added {amount}, charged a {fee} fee")

class SavingsAccountReplaced(Account):
    def deposit(self, amount):
        self.balance += amount + 10   # duplicated the OLD logic, ignoring the fee entirely

class SavingsAccountExtended(Account):
    def deposit(self, amount):
        super().deposit(amount)        # always uses Account's CURRENT logic
        self.balance += 10

r = SavingsAccountReplaced("Kavya", 1000)
e = SavingsAccountExtended("Kavya", 1000)

r.deposit(100)
e.deposit(100)

print("Replaced version balance:", r.balance, "(missed the new fee logic entirely)")
print("Extended version balance:", e.balance, "(automatically picked up the fee)")

  Account.deposit: added 100, charged a 2 fee
Replaced version balance: 1110 (missed the new fee logic entirely)
Extended version balance: 1108 (automatically picked up the fee)


### Concept 4: Multiple inheritance

In [8]:
class Auditable:
    def log_action(self, action):
        print(f"  [LOG] {action}")

class InterestBearing:
    def add_interest(self, rate):
        self.balance += self.balance * rate

class SavingsAccount(Auditable, InterestBearing):   # TWO parents
    def __init__(self, owner, balance):
        self.owner = owner
        self.balance = balance

s = SavingsAccount("Kavya", 1000)

s.log_action("account created")   # from Auditable
s.add_interest(0.05)                # from InterestBearing

print("s.balance:", s.balance)
print("isinstance(s, Auditable):", isinstance(s, Auditable))
print("isinstance(s, InterestBearing):", isinstance(s, InterestBearing))
print(SavingsAccount.__dict__)

  [LOG] account created
s.balance: 1050.0
isinstance(s, Auditable): True
isinstance(s, InterestBearing): True
{'__module__': '__main__', '__firstlineno__': 9, '__init__': <function SavingsAccount.__init__ at 0x10b638880>, '__static_attributes__': ('balance', 'owner'), '__doc__': None}


#### SavingsAccount.__dict__ doesn't contain log_action or add_interest at all — just __init__ and some bookkeeping. That confirms multiple inheritance doesn't copy the parents' methods down into the child's own dict. SavingsAccount doesn't own those methods — it just means "if you can't find something on me, go check Auditable, then InterestBearing" — the exact same fallback lookup pattern you already know from single inheritance, just now with two places to check instead of one.

### MRO (Method Resolution Order)

In [9]:
class A:
    def greet(self):
        print("Hello from A")

class B:
    def greet(self):
        print("Hello from B")

class C(A, B):    # A listed first
    pass

c = C()
c.greet()
print("C.__mro__:", C.__mro__)

Hello from A
C.__mro__: (<class '__main__.C'>, <class '__main__.A'>, <class '__main__.B'>, <class 'object'>)


#### Both A and B have a greet method. c.greet() ran A's version — because A was listed first in class C(A, B):. C.__mro__ shows you the exact order Python checks: C itself first (nothing there), then A (found it, stop), never even reaching B.

In [10]:
class D(B, A):    # B listed first this time
    pass

d = D()
d.greet()
print("D.__mro__:", D.__mro__)

Hello from B
D.__mro__: (<class '__main__.D'>, <class '__main__.B'>, <class '__main__.A'>, <class 'object'>)


MRO is just the extension of the exact lookup rule you already know (instance → class → fall back further) into a precise, ordered list for cases with multiple parents. __mro__ shows you that list directly — Python always searches it left to right, stops at the first match, and the order is determined by how you wrote the parent list in the class definition 

### The diamond problem: two parents sharing a common ancestor, and how Python's C3 linearization resolves it

      Base
      /  \
     A    B
      \  /
       C  



C inherits from both A and B, and both A and B inherit from the same Base. Naively, you'd expect Base to get visited twice if C calls up through both A and B — once via each path. Let's see what Python actually does:

In [2]:
class Base:
    def greet(self):
        print("Hello from Base")

class A(Base):
    def greet(self):
        print("Hello from A")
        super().greet()

class B(Base):
    def greet(self):
        print("Hello from B")
        super().greet()

class C(A, B):
    def greet(self):
        print("Hello from C")
        super().greet()

c = C()
c.greet()

for cls in C.__mro__:
    print(" ", cls)

Hello from C
Hello from A
Hello from B
Hello from Base
  <class '__main__.C'>
  <class '__main__.A'>
  <class '__main__.B'>
  <class '__main__.Base'>
  <class 'object'>


#### when A.greet calls super().greet(), you might expect that to go straight to Base (since A's only parent is Base). It doesn't. Look at the output — after "Hello from A" comes "Hello from B", then "Hello from Base". super() isn't "go to my direct parent" — it's "go to whatever's next in the full MRO list." Since the MRO is C → A → B → Base → object, A's super() actually lands on B next, and only B's super() finally reaches Base

##### This ordering — one single, consistent path where every class appears exactly once — is computed by an algorithm called C3 linearization. It's what guarantees that no matter how tangled your inheritance diamond gets, every ancestor gets visited exactly once, in an order that respects both "parents come before their own parents" and "the order you listed the parents in matters." You'll basically never compute this by hand — ClassName.__mro__ just tells you the answer Python already worked out.

### isinstance() and issubclass() across the chain

In [4]:
c = C()   # from the diamond: C(A, B), both inheriting from Base

print("isinstance(c, C):", isinstance(c, C))
print("isinstance(c, A):", isinstance(c, A))
print("isinstance(c, B):", isinstance(c, B))
print("isinstance(c, Base):", isinstance(c, Base))
print("isinstance(c, B):", isinstance(c, B))

print("issubclass(C, A):", issubclass(C, A))
print("issubclass(C, Base):", issubclass(C, Base))
print("issubclass(A, B):", issubclass(A, B))

isinstance(c, C): True
isinstance(c, A): True
isinstance(c, B): True
isinstance(c, Base): True
isinstance(c, B): False
issubclass(C, A): True
issubclass(C, Base): True
issubclass(A, B): False


isinstance(c, X) comes back True for every single class in c's entire ancestor chain — C itself, both direct parents A and B, and the shared grandparent Base. c doesn't just belong to C — it genuinely is an A, is a B, is a Base, all simultaneously, because inheritance is transitive: if C inherits from A, and A inherits from Base, then anything that's a C is automatically also a Base.

issubclass() asks the same kind of question, but between two classes directly, no instance involved: issubclass(C, A) → True (direct parent). issubclass(C, Base) → True (grandparent, still counts). issubclass(A, B) → False — A and B are siblings; they only share Base as a common ancestor, neither one inherits from the other, so there's no subclass relationship between them at all.

### Concept 8: Why this matters — real frameworks

In [13]:
class MiniModule:
    """A tiny stand-in for the SHAPE of nn.Module."""
    def __init__(self):
        self.layers = {}

    def register_layer(self, name, layer):
        self.layers[name] = layer

    def summary(self):
        print(f"{type(self).__name__} has {len(self.layers)} layer(s):")
        for name, layer in self.layers.items():
            print(f"  - {name}: {layer}")

class MyModel(MiniModule):
    def __init__(self):
        super().__init__()                          # get all the shared machinery for free
        self.register_layer("linear1", "Linear(10, 20)")
        self.register_layer("linear2", "Linear(20, 1)")

    def forward(self, x):                            # YOUR specific logic only
        return f"processed '{x}' through {len(self.layers)} layers"

m = MyModel()
m.summary()
print(m.forward("input_data"))

MyModel has 2 layer(s):
  - linear1: Linear(10, 20)
  - linear2: Linear(20, 1)
processed 'input_data' through 2 layers
